In [79]:
import os, json, time, numpy as np, pandas as pd, dash
import plotly.express as px, plotly.graph_objs as go, dash_bootstrap_components as dbc
from dash import Input, Output, State, dcc, html, dash_table, ALL, MATCH, ctx
#region Start
app = dash.Dash(
    __name__,
    external_stylesheets=[
        dbc.themes.BOOTSTRAP, #Wygląd
        "https://cdn.jsdelivr.net/npm/bootstrap-icons@1.11.0/font/bootstrap-icons.css"
    ],
    suppress_callback_exceptions=True
)
DATA_FOLDER = 'Dane Dash/' #Folder do zczytania gotowych danych
frames = []  
for filename in os.listdir(DATA_FOLDER):
    if filename.endswith('.json'):
        filepath = os.path.join(DATA_FOLDER, filename)
        frames.append(pd.read_json(filepath))
df = pd.concat(frames, ignore_index=True)
timestamp = df['timestamp']
timestamp_pl = timestamp.dt.tz_convert('Europe/Warsaw')
df['timestamp_pl'] = timestamp_pl
df['day_of_year'] = timestamp_pl.dt.day_of_year
df['weekday'] = timestamp_pl.dt.day_name('Pl')
df['weekday_number'] = timestamp_pl.dt.weekday
df['date'] = timestamp_pl.dt.date
df['localized_time_pl'] = df['timestamp_pl'].dt.tz_localize(None)

In [80]:
dfAK = df[df.osoba == 'AK']
dfAK.sample(3)

,timestamp,app,status,duration,category,osoba,device,timestamp_pl,day_of_year,weekday,weekday_number,date,localized_time_pl
4856,2026-01-07 11:23:07.512000+00:00,Systemowy menedżer pulpitu,not_afk,0.000,inne,AK,Phone,2026-01-07 12:23:07.512000+01:00,7,Środa,2,2026-01-07,2026-01-07 12:23:07.512
67034,2026-01-07 19:32:17.209000+00:00,chrome.exe,not-afk,83.971,oglądanie filmów,AK,PC,2026-01-07 20:32:17.209000+01:00,7,Środa,2,2026-01-07,2026-01-07 20:32:17.209
63559,2026-01-01 18:36:23.808000+00:00,rstudio.exe,not-afk,57.545,studia,AK,PC,2026-01-01 19:36:23.808000+01:00,1,Czwartek,3,2026-01-01,2026-01-01 19:36:23.808


Znajdźmy czasy pobudki z każdego dnia

In [81]:
poranki = dfAK[(dfAK['localized_time_pl'].dt.hour < 16) & (dfAK['localized_time_pl'].dt.hour > 5)]

In [98]:
pobudki = poranki.groupby('day_of_year').agg(End=('timestamp_pl','min'))
pobudki.sample(3)

,End
day_of_year,
354,2025-12-20 08:30:01.683000+01:00
9,2026-01-09 06:30:00.509000+01:00
7,2026-01-07 06:30:01.594000+01:00


Znajdźmy czasy pójście spać

In [99]:
wieczory = dfAK[(dfAK['localized_time_pl'].dt.hour > 16) | (dfAK['localized_time_pl'].dt.hour < 5)]
wieczory.sample(3)

,timestamp,app,status,duration,category,osoba,device,timestamp_pl,day_of_year,weekday,weekday_number,date,localized_time_pl
56823,2025-12-17 19:44:40.022000+00:00,chrome.exe,not-afk,2.022,inne,AK,PC,2025-12-17 20:44:40.022000+01:00,351,Środa,2,2025-12-17,2025-12-17 20:44:40.022
66595,2026-01-05 22:01:15.063000+00:00,chrome.exe,not-afk,2.021,oglądanie filmów,AK,PC,2026-01-05 23:01:15.063000+01:00,5,Poniedziałek,0,2026-01-05,2026-01-05 23:01:15.063
61168,2025-12-29 20:47:48.529000+00:00,aces.exe,not-afk,19.286,inne,AK,PC,2025-12-29 21:47:48.529000+01:00,363,Poniedziałek,0,2025-12-29,2025-12-29 21:47:48.529


Grupuje po dniach, przy czym godziny przed północy wliczam do następnego dnia

In [100]:
wieczory.loc[wieczory['localized_time_pl'].dt.hour>16,'day_of_year'] += 1
# Z pierszego stycznia staje się zero, więc trzeba dać na 365
wieczory.loc[wieczory['day_of_year'] == 366,'day_of_year'] = 1

In [101]:
zasypianie = wieczory.groupby('day_of_year').agg(Start=('timestamp_pl','max'))

In [102]:
zasypianie

,Start
day_of_year,
1,2026-01-01 01:17:52.732000+01:00
2,2026-01-02 02:12:40.221000+01:00
3,2026-01-02 23:54:52.881000+01:00
4,2026-01-04 02:00:18.444000+01:00
5,2026-01-05 01:41:18.814000+01:00
6,2026-01-05 23:48:00.985000+01:00
7,2026-01-06 23:39:34.603000+01:00
8,2026-01-07 23:47:19.076000+01:00
9,2026-01-09 00:01:56.456000+01:00


In [103]:
pobudki

,End
day_of_year,
1,2026-01-01 09:56:49.206000+01:00
2,2026-01-02 10:14:24.814000+01:00
3,2026-01-03 10:55:28.275000+01:00
4,2026-01-04 11:30:16.855000+01:00
5,2026-01-05 08:30:01.206000+01:00
6,2026-01-06 08:30:00.519000+01:00
7,2026-01-07 06:30:01.594000+01:00
8,2026-01-08 08:27:04.512000+01:00
9,2026-01-09 06:30:00.509000+01:00


In [ ]:
SenAK = pd.concat([zasypianie,pobudki],axis=1).reset_index()

,day_of_year,Start,End
3,4,2026-01-04 02:00:18.444000+01:00,2026-01-04 11:30:16.855000+01:00
15,341,2025-12-07 00:48:46.843000+01:00,2025-12-07 08:00:00.526000+01:00
13,339,2025-12-04 23:44:51.916000+01:00,2025-12-05 08:00:01.236000+01:00


In [124]:
SenAK['Date'] = SenAK['End'].dt.strftime('%d.%m.%Y')
SenAK['StartTime'] = SenAK['Start'].dt.strftime('%H:%M')
SenAK['EndTime'] = SenAK['End'].dt.strftime('%H:%M')
SenAK.sample(3)

,day_of_year,Start,End,Date,StartTime,EndTime
16,342,2025-12-07 22:35:23.033000+01:00,2025-12-08 08:00:00.635000+01:00,08.12.2025,22:35,08:00
39,365,2025-12-30 23:32:15.737000+01:00,2025-12-31 11:08:56.032000+01:00,31.12.2025,23:32,11:08
22,348,2025-12-14 00:14:10.257000+01:00,2025-12-14 08:00:01.514000+01:00,14.12.2025,00:14,08:00


In [128]:
MojSen = SenAK[['Date','StartTime','EndTime']]
MojSen.head()

,Date,StartTime,EndTime
0,01.01.2026,01:17,09:56
1,02.01.2026,02:12,10:14
2,03.01.2026,23:54,10:55
3,04.01.2026,02:00,11:30
4,05.01.2026,01:41,08:30


In [129]:
MojSen.to_csv('SenAK.csv', index=False)